# Evaluate a CRM analyst before acting on its budget advice

A CRM query can run successfully and still reverse a budget recommendation. Joining payments to
contacts can multiply revenue for accounts with several contacts. Comparing young cohorts with
mature ones can also turn incomplete follow-up into a confident conclusion.

This example evaluates a query and a decision separately. It uses a read-only SQLite database,
three synthetic cases, and hand-calculated reference values. The first sections run offline with
Python 3.11 or newer. The optional final section uses Claude through the Messages API.

The decision rule is for this example. Advance to another limited test when the observed
treatment-minus-control net margin is at least USD 100 per assigned account. Wait while any
planned account lacks 30 days of follow-up. Otherwise stop expansion for now. The small
datasets exercise the rule and do not estimate the precision of a real experiment.

## 1. Define the data and the expected answers

The randomization unit is the account. Use every assigned account, including accounts that
never pay. Net margin is the sum of signed settled contributions in the first 30 days, minus
the offer cost at assignment. Day zero is inclusive and day 30 is exclusive. Signed entries
represent payments and refunds. The synthetic assignment and payment dates use UTC dates.

The three cases exercise a one-to-many join, immature follow-up, and a positive control
that should advance. The expected totals below come from the fixture arithmetic and are
kept separate from the model prompt.

In [1]:
import json
import sqlite3
from datetime import date, timedelta
from fractions import Fraction

AS_OF = "2026-11-08"
SCHEMA = """
CREATE TABLE assignments (
    account_id TEXT PRIMARY KEY, arm TEXT, assigned_on TEXT, offer_cost_cents INTEGER
);
CREATE TABLE contacts (contact_id TEXT PRIMARY KEY, account_id TEXT);
CREATE TABLE payments (
    payment_id TEXT PRIMARY KEY, account_id TEXT, settled_on TEXT, contribution_cents INTEGER
);
"""


def database(control, treatment, *, immature=False, fanout=False):
    conn = sqlite3.connect(":memory:")
    conn.row_factory = sqlite3.Row
    conn.executescript(SCHEMA)
    for arm, amounts in (("control", control), ("treatment", treatment)):
        for i, amount in enumerate(amounts):
            account = f"{arm}-{i}"
            assigned = date(2026, 10, 25) if immature and i >= 2 else date(2026, 9, 1)
            cost = 10_000 if arm == "treatment" else 0
            conn.execute(
                "INSERT INTO assignments VALUES (?, ?, ?, ?)", (account, arm, str(assigned), cost)
            )
            n_contacts = 3 if fanout and account == "treatment-0" else 1
            conn.executemany(
                "INSERT INTO contacts VALUES (?, ?)",
                [(f"{account}-{j}", account) for j in range(n_contacts)],
            )
            if fanout and account == "treatment-0":
                events = [(5, amount + 100), (7, -100), (30, 5000), (-1, 2000)]
            else:
                events = [(5, amount)] if amount else []
            conn.executemany(
                "INSERT INTO payments VALUES (?, ?, ?, ?)",
                [
                    (f"{account}-{j}", account, str(assigned + timedelta(days=day)), dollars * 100)
                    for j, (day, dollars) in enumerate(events)
                ],
            )
    conn.commit()
    return conn


cases = {
    "contact_fanout": database([450, 200, 50, 0], [700, 0, 200, 0], fanout=True),
    "immature_cohort": database([300, 100, 0, 0], [1000, 100, 400, 200], immature=True),
    "positive_control": database([100, 100, 0, 0], [1000, 200, 0, 0]),
}
reference = {
    "contact_fanout": {"control": (4, 4, 70_000), "treatment": (4, 4, 50_000)},
    "immature_cohort": {"control": (4, 2, 40_000), "treatment": (4, 2, 90_000)},
    "positive_control": {"control": (4, 4, 20_000), "treatment": (4, 4, 80_000)},
}
expected_decisions = {
    "contact_fanout": "stop",
    "immature_cohort": "wait",
    "positive_control": "advance",
}

## 2. Execute and score the query

Run the same candidate SQL over all three fixtures. Keep query access to the synthetic tables
and a small set of SQLite functions. Bound execution and require one result row per arm.
An application that runs arbitrary model-generated queries against real data needs stronger
isolation and authorization than this local demonstration.

The audited query computes one row per assigned account before it aggregates by arm.
The failing query adds a join to contacts in the payment subquery. Both produce valid SQL
and the same column names.

In [2]:
AUDITED_SQL = """
WITH account_outcomes AS (
    SELECT a.arm, date(a.assigned_on, '+30 days') <= :as_of AS mature,
           COALESCE((
               SELECT SUM(p.contribution_cents)
               FROM payments p
               WHERE p.account_id = a.account_id
                 AND p.settled_on >= a.assigned_on
                 AND p.settled_on < date(a.assigned_on, '+30 days')
           ), 0) - a.offer_cost_cents AS net_margin
    FROM assignments a
    WHERE a.assigned_on <= :as_of
)
SELECT arm, COUNT(*) AS n_assigned, SUM(mature) AS n_mature,
       SUM(CASE WHEN mature THEN net_margin ELSE 0 END) AS net_margin_cents
FROM account_outcomes
GROUP BY arm
"""
FANOUT_SQL = AUDITED_SQL.replace(
    "FROM payments p",
    "FROM payments p JOIN contacts c ON c.account_id = p.account_id",
)
COLUMNS = ("arm", "n_assigned", "n_mature", "net_margin_cents")
ALLOWED_FUNCTIONS = {
    "coalesce",
    "count",
    "date",
    "ifnull",
    "julianday",
    "min",
    "max",
    "nullif",
    "sum",
    "total",
}


def authorize(action, arg1, arg2, database_name, source):
    allowed = (
        action == sqlite3.SQLITE_SELECT
        or (action == sqlite3.SQLITE_READ and arg1 in {"assignments", "contacts", "payments"})
        or (action == sqlite3.SQLITE_FUNCTION and arg2 in ALLOWED_FUNCTIONS)
    )
    return sqlite3.SQLITE_OK if allowed else sqlite3.SQLITE_DENY


def execute(conn, sql):
    steps_left = 50

    def progress():
        nonlocal steps_left
        steps_left -= 1
        return int(steps_left <= 0)

    conn.set_authorizer(authorize)
    conn.set_progress_handler(progress, 1000)
    try:
        cursor = conn.execute(sql, {"as_of": AS_OF})
        if (
            cursor.description is None
            or len(cursor.description) != len(COLUMNS)
            or {column[0] for column in cursor.description} != set(COLUMNS)
        ):
            raise ValueError("Expected arm, n_assigned, n_mature, net_margin_cents")
        rows = cursor.fetchmany(3)
        if len(rows) != 2 or {row["arm"] for row in rows} != {"control", "treatment"}:
            raise ValueError("Expected exactly one result row per arm")
        metrics = {row["arm"]: tuple(row[col] for col in COLUMNS[1:]) for row in rows}
        if any(type(value) is not int for row in metrics.values() for value in row):
            raise ValueError("Counts and net margin must be integers")
        if any(
            assigned <= 0 or mature < 0 or mature > assigned
            for assigned, mature, _ in metrics.values()
        ):
            raise ValueError("Invalid assignment or maturity counts")
        return metrics
    finally:
        conn.set_progress_handler(None, 0)
        conn.set_authorizer(None)


def effect(metrics):
    return Fraction(metrics["treatment"][2], metrics["treatment"][0]) - Fraction(
        metrics["control"][2], metrics["control"][0]
    )


def decision(metrics):
    if any(assigned != mature for assigned, mature, _ in metrics.values()):
        return "wait"
    return "advance" if effect(metrics) >= 10_000 else "stop"


def score(name, metrics, proposed_decision):
    return {
        "metrics_correct": metrics == reference[name],
        "decision_correct": proposed_decision == expected_decisions[name],
    }


for name, conn in cases.items():
    audited = execute(conn, AUDITED_SQL)
    checks = score(name, audited, decision(audited))
    assert all(checks.values()), (name, checks)
    if decision(audited) == "wait":
        print(
            f"{name}: wait ({audited['treatment'][1]} of {audited['treatment'][0]} treatment accounts mature)"
        )
    else:
        print(f"{name}: {decision(audited)}, USD {float(effect(audited)) / 100:+.2f} per account")

fanout = execute(cases["contact_fanout"], FANOUT_SQL)
assert effect(fanout) == 30_000 and decision(fanout) == "advance"
assert score("contact_fanout", fanout, decision(fanout)) == {
    "metrics_correct": False,
    "decision_correct": False,
}
print(f"Contact join: {decision(fanout)}, USD {float(effect(fanout)) / 100:+.2f} per account")

contact_fanout: stop, USD -50.00 per account
immature_cohort: wait (2 of 4 treatment accounts mature)
positive_control: advance, USD +150.00 per account
Contact join: advance, USD +300.00 per account


The contact join multiplies treatment-account payments by the number of contacts.
Account counts and costs stay fixed, so the flawed query reports **+USD 300** per account
and advances. The audited query reports **−USD 50** and stops expansion. The mature
subset of the second fixture looks favorable, but the agreed follow-up rule yields
**wait**. The positive control advances at **+USD 150**.

Report the calculation check and the action check separately. An analyst can choose the
right action from a wrong calculation, or give the wrong action after calculating the
metrics correctly.

## 3. Optional Claude run

The model proposes SQL without seeing the reference results. Execute that SQL locally
against the three fixtures. Give Claude the computed results and ask it to apply the
same decision policy. The two stages expose whether an error arose in the query or the
business interpretation.

Set `RUN_LIVE = True` and provide an Anthropic API key only to run this optional section.
It uses two Messages API calls. The offline checks above need neither a key nor network access.

In [3]:
QUERY_PROMPT = f"""
You are analyzing a randomized CRM offer experiment. Use SQLite.
Schema:
{SCHEMA}
Write one read-only query that works across fixtures. Bind the snapshot date as :as_of.
Return exactly one row per arm with columns arm, n_assigned, n_mature, net_margin_cents.
Include all accounts assigned by the snapshot date in n_assigned.
An account is mature once 30 calendar days have elapsed after assignment.
For mature accounts, sum signed payment contributions settled on or after assignment
and strictly before day 30. Subtract the offer cost once per mature account, including
accounts with no payments. net_margin_cents is the sum of those mature-account margins.
Use integer counts and cents. Return the SQL in the structured response.
"""

DECISION_PROMPT = """
For each case, wait if any planned account is immature. Otherwise compute treatment
minus control net margin per assigned account. Advance to another limited test at or
above 10000 cents per account. Stop expansion otherwise. Use the supplied metrics.
"""
QUERY_SCHEMA = {
    "type": "object",
    "properties": {"sql": {"type": "string"}},
    "required": ["sql"],
    "additionalProperties": False,
}
CASE_IDS = {name: f"case_{i}" for i, name in enumerate(cases, start=1)}
DECISION_SCHEMA = {
    "type": "object",
    "properties": {
        case_id: {"type": "string", "enum": ["advance", "wait", "stop"]}
        for case_id in CASE_IDS.values()
    },
    "required": list(CASE_IDS.values()),
    "additionalProperties": False,
}


def evaluate_model(generate):
    candidate = generate(QUERY_PROMPT, QUERY_SCHEMA)
    try:
        metrics = {name: execute(conn, candidate["sql"]) for name, conn in cases.items()}
    except (sqlite3.Error, ValueError) as exc:
        return {"sql": candidate["sql"], "query_error": str(exc)}
    prompt = DECISION_PROMPT + "\nMetrics ordered as (n_assigned, n_mature, net_margin_cents):\n"
    blind_metrics = {CASE_IDS[name]: row for name, row in metrics.items()}
    blind_answers = generate(prompt + json.dumps(blind_metrics), DECISION_SCHEMA)
    answers = {name: blind_answers[CASE_IDS[name]] for name in cases}
    return {
        "sql": candidate["sql"],
        "metrics": metrics,
        "decisions": answers,
        "checks": {name: score(name, metrics[name], answers[name]) for name in cases},
    }


RUN_LIVE = False
if RUN_LIVE:
    import anthropic

    client = anthropic.Anthropic()

    def generate(prompt, schema):
        response = client.messages.create(
            model="claude-sonnet-5-5",
            max_tokens=1400,
            messages=[{"role": "user", "content": prompt}],
            output_config={"format": {"type": "json_schema", "schema": schema}},
        )
        if response.stop_reason != "end_turn":
            raise RuntimeError(f"Unexpected stop reason: {response.stop_reason}")
        return json.loads("".join(block.text for block in response.content if block.type == "text"))

    print(json.dumps(evaluate_model(generate), indent=2))

The cases are small by design. For a production evaluation, add fixtures drawn from common
errors and novel combinations, keep a held-out set, and verify that database dialects and
permissions match your application. A real decision also needs an analysis of uncertainty
and an experiment design capable of estimating the effect you care about.